In [1]:
!pip install pyspark

In [2]:
import sqlite3
import pandas as pd
import json
import requests

df = pd.read_csv("week12_transactions.csv")
print(df.shape)
df.head()

(2000, 7)


,txn_id,date,hour,district,txn_type,amount_rwf,flagged
0,T00001,2026-11-01,8,Gasabo,send,1000,0
1,T00002,2026-11-09,12,Nyarugenge,send,25000,0
2,T00003,2026-11-19,8,Muhanga,send,51000,0
3,T00004,2026-11-01,17,Huye,send,79000,0
4,T00005,2026-11-15,12,Huye,send,109000,0


In [3]:
conn = sqlite3.connect("momo.db")
df.to_sql("transactions", conn, if_exists="replace", index=False)
print("Loaded", pd.read_sql("SELECT COUNT(*) AS n FROM transactions", conn).iloc[0,0], "rows")

Loaded 2000 rows


In [4]:
#1.1
q11 = """
SELECT txn_id, district, amount_rwf, flagged
FROM transactions
WHERE txn_type = 'send' AND amount_rwf > 100000
ORDER BY amount_rwf DESC
LIMIT 5;
"""
print(pd.read_sql(q11, conn))

q11_count = """
SELECT COUNT(*) AS flagged_count
FROM transactions
WHERE txn_type = 'send' AND amount_rwf > 100000 AND flagged = 1;
"""
print(pd.read_sql(q11_count, conn))

   txn_id    district  amount_rwf  flagged
0  T01708    Kicukiro      140000        0
1  T00364        Huye      139000        0
2  T00895     Muhanga      139000        0
3  T00899  Nyarugenge      139000        0
4  T01314      Gasabo      139000        0
   flagged_count
0             11


In [5]:
#1.2
q12a = """
SELECT txn_type,
       COUNT(*)        AS n,
       SUM(amount_rwf) AS total,
       AVG(amount_rwf) AS avg
FROM transactions
GROUP BY txn_type
ORDER BY total DESC;
"""
print(pd.read_sql(q12a, conn))

q12b = """
SELECT district, SUM(amount_rwf) AS total
FROM transactions
GROUP BY district
ORDER BY total DESC;
"""
print(pd.read_sql(q12b, conn))

  txn_type    n     total           avg
0     send  868  62049000  71485.023041
1  paybill  330  29970000  90818.181818
2  receive  489  27024000  55263.803681
3  airtime  313    833000   2661.341853
     district     total
0     Muhanga  21076000
1      Gasabo  20738000
2        Huye  18736000
3  Nyarugenge  18091000
4    Kicukiro  11338000
5     Musanze  10490500
6      Rubavu  10110500
7   Nyagatare   9296000


In [6]:
#1.3
q13 = """
SELECT hour, COUNT(*) AS n
FROM transactions
GROUP BY hour
ORDER BY n DESC;
"""
sql_hour = pd.read_sql(q13, conn)
print(sql_hour.head())

pandas_hour = (df.groupby("hour").size()
                 .reset_index(name="n")
                 .sort_values("n", ascending=False))
print(pandas_hour.head())

# Confirm they agree
merged = sql_hour.merge(pandas_hour, on="hour", suffixes=("_sql", "_pd"))
print("All agree:", (merged.n_sql == merged.n_pd).all())

   hour    n
0    20  185
1    17  183
2    18  176
3    12  170
4    11  169
    hour    n
20    20  185
17    17  183
18    18  176
12    12  170
11    11  169
All agree: True


In [10]:
df["date"] = pd.to_datetime(df["date"])

In [11]:
#2.1
json_string = '''
{
  "city": "Kigali",
  "readings": [22.1, 24.5, 27.3, 25.0, 21.8]
}
'''
parsed = json.loads(json_string)
temps = parsed["readings"]
print("City:", parsed["city"])
print("Number of readings:", len(temps))
print("Average: {:.2f}".format(sum(temps)/len(temps)))

City: Kigali
Number of readings: 5
Average: 24.14


In [12]:
#2.2
try:
    url = ("https://api.open-meteo.com/v1/forecast"
           "?latitude=-1.9441&longitude=30.0619"
           "&hourly=temperature_2m&forecast_days=1&timezone=Africa%2FKigali")
    data = requests.get(url, timeout=10).json()
    print("Live API used")
except Exception:
    with open("week12_api_sample.json") as f:
        data = json.load(f)
    print("Offline sample used")

times = data["hourly"]["time"]
temps = data["hourly"]["temperature_2m"]

print("Hours:", len(temps))
print("Max temp:", max(temps))
print("Min temp:", min(temps))
print("Hottest hour:", times[temps.index(max(temps))])

Live API used
Hours: 24
Max temp: 24.5
Min temp: 16.3
Hottest hour: 2026-11-22T15:00


In [13]:
#2.3
weather = pd.DataFrame({"time": times, "temp_c": temps})
print(weather.describe())

hot = weather[weather.temp_c >= 25]
print("Hours >= 25°C:", len(hot))

          temp_c
count  24.000000
mean   19.708333
std     2.826108
min    16.300000
25%    17.025000
50%    19.250000
75%    21.750000
max    24.500000
Hours >= 25°C: 0


In [14]:
from pyspark.sql import SparkSession
spark = (SparkSession.builder
         .appName("week12")
         .master("local[*]")
         .getOrCreate())

sdf = spark.read.csv("week12_transactions.csv", header=True, inferSchema=True)

In [15]:
#3.1
print("Rows:", sdf.count())     # expect 2000
sdf.printSchema()
sdf.show(5)

Rows: 2000
root
 |-- txn_id: string (nullable = true)
 |-- date: date (nullable = true)
 |-- hour: integer (nullable = true)
 |-- district: string (nullable = true)
 |-- txn_type: string (nullable = true)
 |-- amount_rwf: integer (nullable = true)
 |-- flagged: integer (nullable = true)

+------+----------+----+----------+--------+----------+-------+
|txn_id|      date|hour|  district|txn_type|amount_rwf|flagged|
+------+----------+----+----------+--------+----------+-------+
|T00001|2026-11-01|   8|    Gasabo|    send|      1000|      0|
|T00002|2026-11-09|  12|Nyarugenge|    send|     25000|      0|
|T00003|2026-11-19|   8|   Muhanga|    send|     51000|      0|
|T00004|2026-11-01|  17|      Huye|    send|     79000|      0|
|T00005|2026-11-15|  12|      Huye|    send|    109000|      0|
+------+----------+----+----------+--------+----------+-------+
only showing top 5 rows


In [16]:
#3.2
sdf.createOrReplaceTempView("transactions")

above = sdf.filter(sdf.amount_rwf > 100000)
print("Above 100k:", above.count())   # expect 451
above.show(5)

Above 100k: 451
+------+----------+----+--------+--------+----------+-------+
|txn_id|      date|hour|district|txn_type|amount_rwf|flagged|
+------+----------+----+--------+--------+----------+-------+
|T00005|2026-11-15|  12|    Huye|    send|    109000|      0|
|T00009|2026-11-01|   8|  Gasabo|    send|    109000|      0|
|T00011|2026-11-22|  18|  Rubavu|    send|    102000|      0|
|T00012|2026-11-20|   9| Musanze| receive|    112000|      0|
|T00016|2026-11-03|  19| Muhanga| paybill|    154000|      0|
+------+----------+----+--------+--------+----------+-------+
only showing top 5 rows


In [17]:
#3.3
from pyspark.sql.functions import sum as _sum, count, avg

by_type = (sdf.groupBy("txn_type")
              .agg(count("*").alias("n"),
                   _sum("amount_rwf").alias("total"),
                   avg("amount_rwf").alias("avg")))
by_type.show()

by_district = (sdf.groupBy("district")
                  .agg(_sum("amount_rwf").alias("total"))
                  .orderBy("total", ascending=False))
by_district_pd = by_district.toPandas()
print(by_district_pd)

# Cross-check with Pandas (equivalent to SQL)
pd_check = (df.groupby("txn_type")["amount_rwf"]
              .agg(["count","sum","mean"]).reset_index())
print(pd_check)

+--------+---+--------+------------------+
|txn_type|  n|   total|               avg|
+--------+---+--------+------------------+
| paybill|330|29970000| 90818.18181818182|
| airtime|313|  833000|2661.3418530351437|
| receive|489|27024000|  55263.8036809816|
|    send|868|62049000| 71485.02304147465|
+--------+---+--------+------------------+

     district     total
0     Muhanga  21076000
1      Gasabo  20738000
2        Huye  18736000
3  Nyarugenge  18091000
4    Kicukiro  11338000
5     Musanze  10490500
6      Rubavu  10110500
7   Nyagatare   9296000
  txn_type  count       sum          mean
0  airtime    313    833000   2661.341853
1  paybill    330  29970000  90818.181818
2  receive    489  27024000  55263.803681
3     send    868  62049000  71485.023041


In [18]:
#BONUS
spark.sql("""
SELECT txn_type,
       COUNT(*)        AS n,
       SUM(amount_rwf) AS total,
       AVG(amount_rwf) AS avg
FROM transactions
GROUP BY txn_type
ORDER BY total DESC
""").show()

+--------+---+--------+------------------+
|txn_type|  n|   total|               avg|
+--------+---+--------+------------------+
|    send|868|62049000| 71485.02304147465|
| paybill|330|29970000| 90818.18181818182|
| receive|489|27024000|  55263.8036809816|
| airtime|313|  833000|2661.3418530351437|
+--------+---+--------+------------------+



In [ ]:
#4


Scenario 1

— A 200-row CSV of student marks you need to summarize tonight.**
→ **Pandas.** At 200 rows the whole file fits in memory, and a `groupby` answers the question in one line — spinning up SQLite or Spark would add setup cost for no benefit.

Scenario 2

 — A 40 GB transaction log stored in a data lake, aggregated by district.**
→ **Spark.** It distributes the `GROUP BY` across a cluster and reads from the lake natively, which Pandas and SQLite cannot do at that size.

Scenario 3

— A business analyst wants a repeatable weekly report of totals by type from a relational database.**
→ **SQL.** The aggregation runs next to the data as a declarative, schedulable query, so the analyst gets the same result every week without maintaining Python code.

Scenario 4

— You need live weather for a dashboard.**
→ **API.** The data changes continuously and lives outside your systems, so an HTTP call fetches fresh values that you then cache for the dashboard.